# Wasserstein Ascend Descend
### (C. and N.) Garcia Trillos

In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
import sys
import numpy as np
from Robust_nn.WAD import WAD2scale
import matplotlib.pyplot as plt
from utils.utils import read_vision_dataset
from utils.convnet import ConvNet
from utils.convnet_silu import ConvNetSiLU
import os
import torch
import gc
import torch.nn as nn
 

**Read the DataLoader**

In [9]:
trainloader, testloader = read_vision_dataset('../data', dataset='MNIST')

In [10]:
# Create some networks

n_nets = 5
net_lst = [ConvNet() for i in range(n_nets)]


In [72]:
adv_net = WAD2scale(net_list = net_lst, trainloader=trainloader, testloader = testloader, device = None , criterion= nn.CrossEntropyLoss(), scale_factor=1)
adv_net.set_optimizer()
adv_net.train(epochs = 2)


Epoch: 0
Input list shape: torch.Size([6, 128, 1, 28, 28]) Inputs shape: torch.Size([128, 1, 28, 28]) tagets shape: torch.Size([128])
Inner: 0|tensor([0.1654, 0.1678, 0.1682, 0.1669, 0.1661, 0.1657])
Inner: 1|tensor([0.1642, 0.1687, 0.1697, 0.1672, 0.1655, 0.1648])
i,j: 0 0
i,j: 0 1
i,j: 0 2
i,j: 0 3
i,j: 0 4
i,j: 0 5
i,j: 1 0
i,j: 1 1
i,j: 1 2
i,j: 1 3
i,j: 1 4
i,j: 1 5
i,j: 2 0
i,j: 2 1
i,j: 2 2
i,j: 2 3
i,j: 2 4
i,j: 2 5
i,j: 3 0
i,j: 3 1
i,j: 3 2
i,j: 3 3
i,j: 3 4
i,j: 3 5
i,j: 4 0
i,j: 4 1
i,j: 4 2
i,j: 4 3
i,j: 4 4
i,j: 4 5
Input list shape: torch.Size([6, 128, 1, 28, 28]) Inputs shape: torch.Size([128, 1, 28, 28]) tagets shape: torch.Size([128])
Inner: 0|tensor([0.1645, 0.1687, 0.1696, 0.1681, 0.1642, 0.1648])
Inner: 1|tensor([0.1647, 0.1685, 0.1697, 0.1691, 0.1629, 0.1650])
i,j: 0 0
i,j: 0 1
i,j: 0 2
i,j: 0 3
i,j: 0 4
i,j: 0 5
i,j: 1 0
i,j: 1 1
i,j: 1 2


KeyboardInterrupt: 

In [51]:
tensor1 = torch.randn(1,10)
tensor2 = torch.randn(10,20)
torch.matmul(tensor1, tensor2).size()

torch.Size([1, 20])

In [33]:
inp,targ = next(iter(testloader))

In [35]:
net_lst[0](inp).shape

torch.Size([128, 10])

**Creating and comparing results**

In [ ]:
# options = {'only o1':(True,False, False), 'only o2':(False, True, False), 'both': (True,True, False), 'none':(False,False,False), 'modO2':(False,True,True) } 
options = {'only o1':(True,False, False), 'both': (True,True, True), 'none':(False,False,False) } 
rvec = [2,4,np.inf]
deltav = [0.2]
mdict = {}
basepath = os.curdir
mpath = os.path.join(basepath,'models')

for i,(k,(o1,o2, mod_o2)) in enumerate(options.items()):
  auxd = {}
  for r in rvec:
    rstr = str(r)
    print('\n*******\n ** Case '+k+', r=',r,'\n ******')
    torch.manual_seed(0)
    np.random.seed(0)
    auxd[rstr] = {}
    network = ConvNet()
    net_RegTrin = OrdTwoL(network, trainloader, testloader,  device='cuda', delta =0.2, r= r, o2=o2, o1=o1, mod_o2=mod_o2) #'cuda'
    net_RegTrin.set_optimizer(optim_alg='Adam', args={'lr':1e-4})
    net_RegTrin.train(epochs=5, delta=deltav)
    auxd[rstr]['train_loss'] = net_RegTrin.train_loss.copy()
    auxd[rstr]['train_acc'] = net_RegTrin.train_acc.copy()
    auxd[rstr]['train_reg'] = net_RegTrin.train_reg.copy()
    auxd[rstr]['test_acc_adv'] = net_RegTrin.test_acc_adv.copy()
    auxd[rstr]['test_acc_clean'] = net_RegTrin.test_acc_clean.copy()
    auxd[rstr]['train_times'] = net_RegTrin.train_times.copy()
    torch.save(network.state_dict(), os.path.join(mpath, k+'_r_'+str(r)+'.pth' ))
    del(network)
    gc.collect()
    torch.cuda.empty_cache()
  mdict[k] = auxd
  with open('tests_all.txt','w') as data: 
      data.write(str(mdict))